<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_GEMMA_INKLING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://openrouter.ai/settings/credits

https://huggingface.co/frankmorales2020/gemma-4-e4b-stl10-topo-2026

In [2]:
!pip install -q transformers torch openai python-dotenv pyyaml requests numpy pydantic fastapi uvicorn
!pip install -U bitsandbytes>=0.46.1 -q
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
import requests
import json

def list_openrouter_models():
    """
    Connects to the OpenRouter API to fetch the list of available models
    and prints key details for each model.
    """

    # OpenRouter API endpoint for listing models
    API_URL = "https://openrouter.ai/api/v1/models"

    print(f"Fetching models from: {API_URL}\n")

    try:
        # Send a GET request to the models endpoint
        response = requests.get(API_URL)

        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # The list of models is contained in the 'data' field
        models = data.get('data', [])

        if not models:
            print("No models found in the API response.")
            return

        # --- Print the results in a formatted way ---
        print(f"Found {len(models)} total models. Key details:\n")
        print("{:<45} {:<30} {:<15}".format("Model ID", "Name", "Context Length"))
        print("-" * 90)

        # Iterate over the model list and print details
        for model in models:
            model_id = model.get('id', 'N/A')
            name = model.get('name', 'N/A')
            context_length = model.get('context_length', 'N/A')

            # Truncate model ID and name for clean printing
            display_id = model_id[:42] + '...' if len(model_id) > 45 else model_id
            display_name = name[:27] + '...' if len(name) > 30 else name

            print("{:<45} {:<30} {:<15}".format(display_id, display_name, context_length))

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while connecting to the OpenRouter API: {e}")
    except json.JSONDecodeError:
        print("Error: Failed to decode JSON response from the API.")

if __name__ == "__main__":
    list_openrouter_models()

In [2]:
# ============================================================
# INKLING API TEST - Standalone
# Tests OpenRouter access to Thinking Machines Inkling
# ============================================================

import requests
import json

# ------------------------------------------------------------------
# Load API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not API_KEY:
        raise ValueError("OPENROUTER_API_KEY not found in Colab secrets.")
    print(f"✅ API key loaded! Ending: ****{API_KEY[-4:]}")
except Exception as e:
    print(f"❌ Failed to load API key: {e}")
    API_KEY = input("🔑 Enter your OpenRouter API key manually: ")

# ------------------------------------------------------------------
# Test Inkling
# ------------------------------------------------------------------

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# ✅ CORRECT FORMAT - reasoning as object with enabled: true
payload = {
    "model": "thinkingmachines/inkling",
    "messages": [
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ],
    "temperature": 0.7,
    "max_tokens": 100,
    "reasoning": {"enabled": True}  # ✅ CORRECT: object, not boolean
}

print("\n" + "="*60)
print("📤 Sending request to Inkling...")
print("="*60)

print(f"\n📋 Payload:")
print(json.dumps(payload, indent=2))

try:
    response = requests.post(url, headers=headers, json=payload, timeout=30)

    print(f"\n📥 Status Code: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        content = data['choices'][0]['message']['content']
        print(f"\n✅ SUCCESS!")
        print(f"📝 Response: {content}")
        print(f"\n📦 Full response:")
        print(json.dumps(data, indent=2)[:500] + "...")
    else:
        print(f"\n❌ ERROR:")
        print(f"   Status: {response.status_code}")
        print(f"   Response: {response.text[:500]}")

except Exception as e:
    print(f"\n❌ Request failed: {e}")

print("\n" + "="*60)

✅ API key loaded! Ending: ****f808

📤 Sending request to Inkling...

📋 Payload:
{
  "model": "thinkingmachines/inkling",
  "messages": [
    {
      "role": "user",
      "content": "What is the capital of France? Answer in one word."
    }
  ],
  "temperature": 0.7,
  "max_tokens": 100,
  "reasoning": {
    "enabled": true
  }
}

📥 Status Code: 200

✅ SUCCESS!
📝 Response: Paris

📦 Full response:
{
  "id": "gen-1785836690-J0OcaN864GUwYIT9mYqW",
  "object": "chat.completion",
  "created": 1785836690,
  "model": "thinkingmachines/inkling",
  "provider": "DeepInfra",
  "system_fingerprint": null,
  "service_tier": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "Paris",
        "refusal": null,
        "reasoning": "The user is asking f...



In [1]:
# ============================================================
# FERRARI AI: Gemma-4 E4B TOPO-2026 (STL-10 Certified) + Inkling
# UPDATED: Increased Inkling output tokens to 2048
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# CORRECT INKLING MODEL ID
# ============================================================
INKLING_MODEL_ID = "thinkingmachines/inkling"

# ============================================================
# INKLING CONFIGURATION - INCREASED TOKENS
# ============================================================
INKLING_MAX_TOKENS = 2048  # ✅ Increased from 500 to 2048
INKLING_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ============================================================
# PART 2: INKLING CLIENT - INCREASED MAX TOKENS
# ============================================================

@dataclass
class InklingResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class InklingClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = INKLING_MODEL_ID,
                 max_tokens: int = INKLING_MAX_TOKENS,
                 temperature: float = INKLING_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Inkling will not work.")
        else:
            print(f"✅ Inkling client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> InklingResponse:
        """
        Send a query to Inkling via OpenRouter.

        Args:
            prompt: User query
            temperature: Override temperature (0-1)
            max_tokens: Override max output tokens
        """
        if not self.api_key:
            return InklingResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        # Use defaults or overrides
        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        # ✅ CORRECT FORMAT - reasoning as object with enabled: true
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60  # Increased timeout for longer responses
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return InklingResponse(content=error_content)

            data = response.json()

            # Extract content and reasoning properly
            message = data['choices'][0]['message']

            # Get content - could be None, use reasoning as fallback
            content = message.get('content')
            reasoning = message.get('reasoning')

            # If content is None or empty, use reasoning as content
            if not content and reasoning:
                content = reasoning
                reasoning = None

            # If both are None, set default
            if not content:
                content = "No response content received."

            return InklingResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return InklingResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: SMART ROUTER
# ============================================================

class TaskRouter:
    def __init__(self):
        print("✅ Router initialized")

    def route(self, query: str) -> Dict:
        query_lower = query.lower()

        # Classification keywords -> Gemma
        if any(kw in query_lower for kw in ['animal', 'vehicle']):
            return {'model': 'gemma', 'task': 'A'}
        if any(kw in query_lower for kw in ['natural', 'man-made']):
            return {'model': 'gemma', 'task': 'B'}
        if any(kw in query_lower for kw in ['living', 'non-living', 'alive']):
            return {'model': 'gemma', 'task': 'C'}
        if any(kw in query_lower for kw in ['classify', 'category', 'label', 'type of', 'is this']):
            return {'model': 'gemma', 'task': 'C'}

        # Everything else -> Inkling
        return {'model': 'inkling', 'task': None}


# ============================================================
# PART 4: FERRARI AI ORCHESTRATOR
# ============================================================

class FerrariAI:
    def __init__(self, inkling_max_tokens: int = INKLING_MAX_TOKENS):
        print("="*60)
        print("🏎️  FERRARI AI - Initializing...")
        print("="*60)

        self.gemma = GemmaTOPOCertified()
        self.inkling = InklingClient(max_tokens=inkling_max_tokens)
        self.router = TaskRouter()

        print("\n" + "="*60)
        print("✅ Ferrari AI Ready!")
        print(f"   🧠 Gemma-4 TOPO-2026: CF-Free (0% Forgetting)")
        print(f"      Tasks: Animal/Vehicle, Natural/Man-Made, Living/Non-Living")
        print(f"   🧠 Inkling: Advanced Reasoning")
        print(f"      Model ID: {INKLING_MODEL_ID}")
        print(f"      Max Tokens: {inkling_max_tokens}")
        if OPENROUTER_API_KEY:
            print("   ✅ Inkling API: Connected")
        print("="*60 + "\n")

    def process(self, query: str, max_tokens: Optional[int] = None) -> Dict:
        """
        Process a query through the optimal model.

        Args:
            query: User query
            max_tokens: Override max tokens for Inkling (Gemma ignores this)
        """
        print(f"\n📝 Query: {query[:100]}{'...' if len(query) > 100 else ''}")
        print("-" * 50)

        decision = self.router.route(query)
        print(f"🔀 Routing to: {decision['model'].upper()}")
        if decision.get('task'):
            print(f"   Task: {decision['task']}")
        print("-" * 50)

        if decision['model'] == "gemma":
            result = self.gemma.classify(query, task=decision.get('task', 'C'))
            print(f"🧠 Gemma Result:")
            print(f"   Label: {result['label']}")
            print(f"   Confidence: {result['confidence']:.2%}")
            print(f"   CF-Free: {result.get('cf_free', False)}")
            return {'model': 'gemma', 'result': result}
        else:
            # ✅ Pass max_tokens override if provided
            result = self.inkling.query(query, max_tokens=max_tokens)
            content = result.content if result.content else "No response received"
            print(f"🧠 Inkling Result:")
            print(f"   Content: {content[:200]}{'...' if len(content) > 200 else ''}")
            print(f"   Total Length: {len(content)} characters")
            if result.reasoning:
                print(f"   ✅ Reasoning: {result.reasoning[:100]}...")
            return {'model': 'inkling', 'result': result}


# ============================================================
# PART 5: DEMONSTRATION
# ============================================================

def run_demo():
    print("\n" + "="*60)
    print("🏎️  FERRARI AI - Demonstration")
    print("="*60 + "\n")

    # ✅ Initialize with increased token limit
    ferrari = FerrariAI(inkling_max_tokens=INKLING_MAX_TOKENS)

    test_queries = [
        # Gemma tasks
        "Is an airplane a vehicle?",
        "Is a bird natural or man-made?",
        "Is a car living or non-living?",
        # Inkling tasks - will now have longer responses
        "What is the difference between living and non-living things?",
        "Why is catastrophic forgetting a problem in AI? Explain in detail.",
    ]

    for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*60}")
        print(f"Test {i}: {query}")
        print('='*60)

        result = ferrari.process(query)

        print("\n📊 Final Result:")
        print("-" * 40)

        if result['model'] == 'gemma':
            r = result['result']
            print(f"   Classification: {r['label']}")
            print(f"   Confidence: {r['confidence']:.2%}")
            print(f"   Task: {r.get('task_description', 'N/A')}")
            print(f"   CF-Free: {r.get('cf_free', False)}")
            cert = r.get('certification', {})
            print(f"   Certification: {cert.get('status', 'N/A')}")
        else:
            r = result['result']
            content = r.content if r.content else "No response received"
            print(f"   Response Length: {len(content)} characters")
            print(f"   Response: {content[:500]}{'...' if len(content) > 500 else ''}")
            if r.reasoning:
                print(f"   Reasoning: {r.reasoning[:100]}...")

        print("-" * 40)


# ============================================================
# PART 6: INTERACTIVE CHAT
# ============================================================

def interactive_chat():
    print("\n" + "="*60)
    print("🏎️  FERRARI AI - Interactive Chat")
    print("="*60)
    print("Type your query below. Type 'exit' to quit.")
    print(f"\n📝 Inkling Max Tokens: {INKLING_MAX_TOKENS}")
    print("\n📚 Gemma Tasks (TOPO-2026 Certified):")
    print("   A: Animal vs Vehicle")
    print("   B: Natural vs Man-Made")
    print("   C: Living vs Non-Living")
    print("   Other queries → Inkling for reasoning")
    print("-"*60 + "\n")

    ferrari = FerrariAI(inkling_max_tokens=INKLING_MAX_TOKENS)

    while True:
        try:
            query = input("\n💬 You: ").strip()
            if query.lower() in ['exit', 'quit', 'q']:
                print("\n👋 Goodbye!")
                break

            if not query:
                continue

            result = ferrari.process(query)

            print("\n🤖 Response:")
            print("-"*50)

            if result['model'] == 'gemma':
                r = result['result']
                print(f"Classification: {r['label']}")
                print(f"Confidence: {r['confidence']:.2%}")
                print(f"Task: {r.get('task_description', 'N/A')}")
                print(f"CF-Free: {r.get('cf_free', False)}")
            else:
                r = result['result']
                content = r.content if r.content else "No response received"
                print(content)
                print(f"\n📊 Length: {len(content)} characters")
                if r.reasoning:
                    print(f"\n🧠 Reasoning: {r.reasoning[:200]}...")

            print("-"*50)

        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")


# ============================================================
# CONFIGURATION
# ============================================================

config = {
    'gemma_model': 'frankmorales2020/gemma-4-e4b-stl10-topo-2026',
    'gemma_base': 'frankmorales2020/gemma-4-e4b-unesco-optimized',
    'inkling_model': INKLING_MODEL_ID,
    'inkling_max_tokens': INKLING_MAX_TOKENS,
    'inkling_temperature': INKLING_TEMPERATURE,
    'api_key_configured': bool(OPENROUTER_API_KEY),
    'gemma_tasks': {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    },
    'reasoning_format': '{"enabled": true} ✅'
}

print("\n📦 Configuration:")
print(json.dumps(config, indent=2))

# ============================================================
# RUN THE DEMO
# ============================================================

# ✅ RUN THE DEMO
run_demo()

# Uncomment to use interactive chat instead:
# interactive_chat()

✅ API key loaded from Colab secrets! Ending: ****f808

📦 Configuration:
{
  "gemma_model": "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
  "gemma_base": "frankmorales2020/gemma-4-e4b-unesco-optimized",
  "inkling_model": "thinkingmachines/inkling",
  "inkling_max_tokens": 2048,
  "inkling_temperature": 0.7,
  "api_key_configured": true,
  "gemma_tasks": {
    "A": "Animal vs Vehicle",
    "B": "Natural vs Man-Made",
    "C": "Living vs Non-Living"
  },
  "reasoning_format": "{\"enabled\": true} \u2705"
}

🏎️  FERRARI AI - Demonstration

🏎️  FERRARI AI - Initializing...

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Inkling client initialized
   Model: thinkingmachines/inkling
   Max Tokens: 2048
   Temperature: 0.7
✅ Router initialized

✅ Ferrari AI Ready!
   🧠 Gemma-4 TOPO-2026: CF-Free (0% Forgetting)
      Tasks: Animal/Vehicle, Natural/Man-Made, Living/Non-Living
   🧠 Inkling: Advanced Reasoning
      Model ID: thinkingmachines/inkling
      Max Tokens: 2048
   ✅ Inkling API: Connected


Test 1: Is an airplane a vehicle?

📝 Query: Is an airplane a vehicle?
--------------------------------------------------
🔀 Routing to: GEMMA
   Task: A
--------------------------------------------------
🧠 Gemma Result:
   Label: Vehicle
   Confidence: 99.12%
   CF-Free: True

📊 Final Result:
----------------------

## ✈️ Ferrari AI - Airline Operations Control Center (AOCC) Agent

In [1]:
# ============================================================
# FERRARI AI - AIRLINE OPERATIONS CONTROL CENTER (AOCC) AGENT
# Complete Agentic Solution for Airline Operations
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# CORRECT INKLING MODEL ID
# ============================================================
INKLING_MODEL_ID = "thinkingmachines/inkling"
INKLING_MAX_TOKENS = 2048
INKLING_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ============================================================
# PART 2: INKLING CLIENT
# ============================================================

@dataclass
class InklingResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class InklingClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = INKLING_MODEL_ID,
                 max_tokens: int = INKLING_MAX_TOKENS,
                 temperature: float = INKLING_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Inkling will not work.")
        else:
            print(f"✅ Inkling client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> InklingResponse:
        if not self.api_key:
            return InklingResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return InklingResponse(content=error_content)

            data = response.json()
            message = data['choices'][0]['message']

            content = message.get('content')
            reasoning = message.get('reasoning')

            if not content and reasoning:
                content = reasoning
                reasoning = None

            if not content:
                content = "No response content received."

            return InklingResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return InklingResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: AIRLINE OPERATIONS CONTROL CENTER AGENT
# ============================================================

class AOCCTaskType(Enum):
    FLIGHT_STATUS = "flight_status"
    WEATHER_CHECK = "weather_check"
    CREW_CHECK = "crew_check"
    MAINTENANCE = "maintenance"
    PASSENGER_QUERY = "passenger_query"
    DISRUPTION = "disruption"
    FUEL_CHECK = "fuel_check"
    GENERAL = "general"


@dataclass
class Flight:
    flight_number: str
    origin: str
    destination: str
    departure_time: datetime
    arrival_time: datetime
    status: str
    aircraft: str
    gate: str
    crew: List[str]
    passengers: int

    def to_dict(self) -> Dict:
        return {
            'flight_number': self.flight_number,
            'origin': self.origin,
            'destination': self.destination,
            'departure_time': self.departure_time.strftime('%Y-%m-%d %H:%M'),
            'arrival_time': self.arrival_time.strftime('%Y-%m-%d %H:%M'),
            'status': self.status,
            'aircraft': self.aircraft,
            'gate': self.gate,
            'crew': self.crew,
            'passengers': self.passengers
        }


@dataclass
class WeatherAlert:
    location: str
    severity: str  # LOW, MEDIUM, HIGH, CRITICAL
    description: str
    timestamp: datetime
    impact: str


@dataclass
class AOCCResponse:
    task_type: AOCCTaskType
    action_taken: str
    data: Dict
    reasoning: str
    confidence: float
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class AirlineOperationsAgent:
    """
    Agentic AI System for Airline Operations Control Center (AOCC)
    Combines Gemma-4 classification with Inkling reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("✈️  AIRLINE OPERATIONS CONTROL CENTER AGENT")
        print("="*60)

        # Initialize Ferrari AI components
        self.gemma = GemmaTOPOCertified()
        self.inkling = InklingClient()

        # Flight database (simulated)
        self.flights: Dict[str, Flight] = {}
        self.weather_alerts: List[WeatherAlert] = []
        self._initialize_flight_data()
        self._initialize_weather_data()

        print("="*60)
        print("✅ AOCC Agent Ready!")
        print(f"   📊 Flights: {len(self.flights)}")
        print(f"   🌤️ Weather Alerts: {len(self.weather_alerts)}")
        print("="*60 + "\n")

    def _initialize_flight_data(self):
        """Initialize simulated flight data."""
        now = datetime.now()

        flights_data = [
            ("AA101", "JFK", "LHR", now + timedelta(hours=2), now + timedelta(hours=8), "ON_TIME", "B787", "A12", ["Capt. Smith", "F/O Jones", "S/A Green"], 250),
            ("AA102", "LHR", "JFK", now + timedelta(hours=3), now + timedelta(hours=9), "DELAYED", "B777", "B8", ["Capt. Davis", "F/O Wilson", "S/A Brown"], 280),
            ("AA103", "JFK", "CDG", now + timedelta(hours=1), now + timedelta(hours=6), "ON_TIME", "A380", "C5", ["Capt. Johnson", "F/O Martinez", "S/A Lee"], 350),
            ("AA104", "CDG", "JFK", now + timedelta(hours=4), now + timedelta(hours=10), "CANCELLED", "B787", "D7", ["Capt. Anderson", "F/O Thompson", "S/A White"], 220),
            ("AA105", "LAX", "NRT", now + timedelta(hours=5), now + timedelta(hours=13), "ON_TIME", "B777", "E3", ["Capt. Taylor", "F/O Moore", "S/A Jackson"], 300),
            ("AA106", "NRT", "LAX", now + timedelta(hours=6), now + timedelta(hours=14), "DELAYED", "B787", "F9", ["Capt. Wilson", "F/O Garcia", "S/A Martinez"], 260),
        ]

        for flight_data in flights_data:
            flight = Flight(
                flight_number=flight_data[0],
                origin=flight_data[1],
                destination=flight_data[2],
                departure_time=flight_data[3],
                arrival_time=flight_data[4],
                status=flight_data[5],
                aircraft=flight_data[6],
                gate=flight_data[7],
                crew=flight_data[8],
                passengers=flight_data[9]
            )
            self.flights[flight.flight_number] = flight

    def _initialize_weather_data(self):
        """Initialize simulated weather alerts."""
        now = datetime.now()

        self.weather_alerts = [
            WeatherAlert(
                location="JFK",
                severity="HIGH",
                description="Heavy thunderstorms expected with strong winds",
                timestamp=now + timedelta(hours=1),
                impact="Potential delays and diversions"
            ),
            WeatherAlert(
                location="LHR",
                severity="MEDIUM",
                description="Light fog expected in the morning",
                timestamp=now + timedelta(hours=3),
                impact="Reduced visibility, minor delays"
            ),
            WeatherAlert(
                location="CDG",
                severity="LOW",
                description="Clear skies, no disruptions",
                timestamp=now + timedelta(hours=2),
                impact="Normal operations"
            ),
        ]

    def identify_task(self, query: str) -> Dict:
        """
        Use Gemma to classify the query type.
        """
        query_lower = query.lower()

        # Classification keywords
        task_mapping = {
            'flight': AOCCTaskType.FLIGHT_STATUS,
            'status': AOCCTaskType.FLIGHT_STATUS,
            'departure': AOCCTaskType.FLIGHT_STATUS,
            'arrival': AOCCTaskType.FLIGHT_STATUS,
            'weather': AOCCTaskType.WEATHER_CHECK,
            'forecast': AOCCTaskType.WEATHER_CHECK,
            'storm': AOCCTaskType.WEATHER_CHECK,
            'crew': AOCCTaskType.CREW_CHECK,
            'pilot': AOCCTaskType.CREW_CHECK,
            'attendant': AOCCTaskType.CREW_CHECK,
            'maintenance': AOCCTaskType.MAINTENANCE,
            'repair': AOCCTaskType.MAINTENANCE,
            'technical': AOCCTaskType.MAINTENANCE,
            'passenger': AOCCTaskType.PASSENGER_QUERY,
            'customer': AOCCTaskType.PASSENGER_QUERY,
            'delay': AOCCTaskType.DISRUPTION,
            'cancellation': AOCCTaskType.DISRUPTION,
            'divert': AOCCTaskType.DISRUPTION,
            'fuel': AOCCTaskType.FUEL_CHECK,
            'gas': AOCCTaskType.FUEL_CHECK,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': AOCCTaskType.GENERAL, 'confidence': 0.6}

    def _get_flight_status(self, flight_number: str) -> Dict:
        """Get flight status from database."""
        if flight_number in self.flights:
            return self.flights[flight_number].to_dict()
        return None

    def _get_weather_alerts(self, location: str = None) -> List[Dict]:
        """Get weather alerts for a location."""
        alerts = []
        for alert in self.weather_alerts:
            if location is None or location in alert.location:
                alerts.append({
                    'location': alert.location,
                    'severity': alert.severity,
                    'description': alert.description,
                    'impact': alert.impact
                })
        return alerts

    def _find_flights_by_location(self, location: str) -> List[Dict]:
        """Find flights by origin or destination."""
        results = []
        for flight in self.flights.values():
            if location in [flight.origin, flight.destination]:
                results.append(flight.to_dict())
        return results

    def process_query(self, query: str) -> AOCCResponse:
        """
        Main processing method for the AOCC Agent.
        """
        print(f"\n✈️ AOCC Query: {query}")
        print("-" * 50)

        # Step 1: Identify task type
        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # Step 2: Retrieve data based on task type
        data = {}
        action_taken = ""

        # Extract flight number if mentioned
        import re
        flight_match = re.search(r'([A-Z]{2,3}\d{1,4})', query.upper())
        flight_number = flight_match.group(1) if flight_match else None

        # Extract location if mentioned
        locations = ['JFK', 'LHR', 'CDG', 'LAX', 'NRT']
        location = None
        for loc in locations:
            if loc in query.upper():
                location = loc
                break

        if task_type == AOCCTaskType.FLIGHT_STATUS:
            if flight_number:
                flight_data = self._get_flight_status(flight_number)
                if flight_data:
                    data['flight'] = flight_data
                    action_taken = f"Retrieved status for flight {flight_number}"
                else:
                    data['error'] = f"Flight {flight_number} not found"
                    action_taken = "Flight lookup failed"
            elif location:
                flights = self._find_flights_by_location(location)
                data['flights'] = flights
                action_taken = f"Retrieved flights for {location}"
            else:
                data['all_flights'] = [f.to_dict() for f in self.flights.values()]
                action_taken = "Retrieved all flights"

        elif task_type == AOCCTaskType.WEATHER_CHECK:
            alerts = self._get_weather_alerts(location)
            data['weather_alerts'] = alerts
            action_taken = f"Retrieved weather alerts for {location if location else 'all locations'}"

        elif task_type == AOCCTaskType.CREW_CHECK:
            if flight_number and flight_number in self.flights:
                flight = self.flights[flight_number]
                data['crew'] = flight.crew
                data['flight'] = flight.flight_number
                action_taken = f"Retrieved crew for flight {flight_number}"
            else:
                data['crew'] = [f.to_dict() for f in self.flights.values() if f.crew]
                action_taken = "Retrieved all crew information"

        elif task_type == AOCCTaskType.MAINTENANCE:
            # Simulate maintenance checks
            maintenance_data = {}
            for f in self.flights.values():
                maintenance_data[f.flight_number] = {
                    'aircraft': f.aircraft,
                    'status': 'OK' if random.random() > 0.2 else 'CHECK_REQUIRED',
                    'last_check': (datetime.now() - timedelta(days=random.randint(1, 30))).strftime('%Y-%m-%d'),
                    'next_check': (datetime.now() + timedelta(days=random.randint(1, 30))).strftime('%Y-%m-%d')
                }
            data['maintenance'] = maintenance_data
            action_taken = "Retrieved maintenance status for fleet"

        elif task_type == AOCCTaskType.DISRUPTION:
            disrupted_flights = [f.to_dict() for f in self.flights.values() if f.status in ['DELAYED', 'CANCELLED']]
            data['disrupted_flights'] = disrupted_flights
            data['weather_impact'] = self._get_weather_alerts()
            action_taken = "Retrieved disruption information"

        elif task_type == AOCCTaskType.FUEL_CHECK:
            fuel_data = {}
            for f in self.flights.values():
                fuel_data[f.flight_number] = {
                    'aircraft': f.aircraft,
                    'fuel_remaining': f"{random.randint(15, 85)}%",
                    'estimated_burn': f"{random.randint(5, 25)} tons",
                    'status': 'OK' if random.random() > 0.1 else 'LOW'
                }
            data['fuel_status'] = fuel_data
            action_taken = "Retrieved fuel status for fleet"

        else:  # GENERAL
            data['message'] = "General inquiry received"
            action_taken = "General query processed"

        # Step 3: Generate reasoning/response using Inkling
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        inkling_response = self.inkling.query(reasoning_prompt)

        print(f"🧠 Inkling Reasoning: {inkling_response.content[:200]}...")

        # Step 4: Build response
        return AOCCResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=inkling_response.content,
            confidence=confidence
        )

    def _generate_reasoning_prompt(self, query: str, task_type: AOCCTaskType, data: Dict) -> str:
        """Generate a prompt for Inkling to reason about the situation."""
        return f"""
        You are an Airline Operations Control Center (AOCC) agent assistant.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clear analysis of the situation
        2. Recommended actions based on the data
        3. Any additional insights or warnings
        4. A concise summary for the operations team

        Your response should be professional, actionable, and data-driven.
        """

    def handle_query(self, query: str) -> str:
        """
        Simplified method to handle a query and return a readable response.
        """
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append(f"✈️ AOCC Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        output.append("-" * 60)
        output.append("📊 Reasoning:")
        output.append(response.reasoning)
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION
# ============================================================

def run_aocc_demo():
    """Demonstrate the AOCC Agent in action."""
    print("\n" + "="*60)
    print("✈️  AIRLINE OPERATIONS CONTROL CENTER AGENT")
    print("   Complete Agentic Solution")
    print("="*60 + "\n")

    agent = AirlineOperationsAgent()

    test_queries = [
        "What is the status of flight AA101?",
        "Show me all flights from JFK",
        "What is the weather like at LHR?",
        "Who is the crew on flight AA103?",
        "Are there any maintenance issues?",
        "What flights are delayed or cancelled?",
        "What is the fuel status for the fleet?",
        "What's the overall situation at JFK?",
    ]

    for i, query in enumerate(test_queries[:5], 1):  # Show first 5
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# PART 5: INTERACTIVE AOCC CHAT
# ============================================================

def interactive_aocc():
    """Interactive chat with the AOCC Agent."""
    print("\n" + "="*60)
    print("✈️  AOCC AGENT - Interactive Operations Center")
    print("="*60)
    print("Type your query below. Type 'exit' to quit.")
    print("")
    print("📚 Sample Queries:")
    print("   • Status of flight AA101")
    print("   • Weather at JFK")
    print("   • Crew on flight AA103")
    print("   • Any maintenance issues?")
    print("   • Show delayed flights")
    print("   • Fuel status for the fleet")
    print("   • What's happening at LHR?")
    print("-"*60 + "\n")

    agent = AirlineOperationsAgent()

    while True:
        try:
            query = input("\n✈️ Ops: ").strip()
            if query.lower() in ['exit', 'quit', 'q']:
                print("\n👋 AOCC Agent signing off!")
                break

            if not query:
                continue

            result = agent.handle_query(query)
            print(result)

        except KeyboardInterrupt:
            print("\n\n👋 AOCC Agent signing off!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")


# ============================================================
# PART 6: CONFIGURATION
# ============================================================

config = {
    'gemma_model': 'frankmorales2020/gemma-4-e4b-stl10-topo-2026',
    'gemma_base': 'frankmorales2020/gemma-4-e4b-unesco-optimized',
    'inkling_model': INKLING_MODEL_ID,
    'inkling_max_tokens': INKLING_MAX_TOKENS,
    'inkling_temperature': INKLING_TEMPERATURE,
    'api_key_configured': bool(OPENROUTER_API_KEY),
    'system': 'Airline Operations Control Center Agent',
    'version': '1.0.0'
}

print("\n📦 Configuration:")
print(json.dumps(config, indent=2))

# ============================================================
# RUN THE AOCC DEMO
# ============================================================

# ✅ RUN THE DEMO
run_aocc_demo()

# Uncomment to use interactive chat instead:
# interactive_aocc()

✅ API key loaded from Colab secrets! Ending: ****f808

📦 Configuration:
{
  "gemma_model": "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
  "gemma_base": "frankmorales2020/gemma-4-e4b-unesco-optimized",
  "inkling_model": "thinkingmachines/inkling",
  "inkling_max_tokens": 2048,
  "inkling_temperature": 0.7,
  "api_key_configured": true,
  "system": "Airline Operations Control Center Agent",
  "version": "1.0.0"
}

✈️  AIRLINE OPERATIONS CONTROL CENTER AGENT
   Complete Agentic Solution


✈️  AIRLINE OPERATIONS CONTROL CENTER AGENT

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Inkling client initialized
   Model: thinkingmachines/inkling
   Max Tokens: 2048
   Temperature: 0.7
✅ AOCC Agent Ready!
   📊 Flights: 6
   🌤️ Weather Alerts: 3


TEST 1

✈️ AOCC Query: What is the status of flight AA101?
--------------------------------------------------
📋 Task Identification: flight_status (confidence: 90.00%)
🧠 Inkling Reasoning: **AOCC FLIGHT STATUS BRIEF — AA101 (JFK → LHR)**

---

### 1. SITUATION ANALYSIS

| Parameter | Data Point | Assessment |
|-----------|-----------|------------|
| **Flight** | AA101 | Active long-haul...
✈️ AOCC Response
📋 Task: flight_status
✅ Action: Retrieved status for flight AA101
🔒 Confidence: 90.00%
--------------------------------------